# Profiling Agent Memory Systems: Bottleneck Analysis & Tests

**Scope**: System-side performance and scalability analysis of LLM-based agent memory systems.
Focus on insertion flow (view maintenance), not retrieval accuracy.

**Reference**: Flow diagrams in [`data_flow_relational.md`](./data_flow_relational.md) — relational algebra notation for Graphiti's `add_episode`, `search`, and `build_communities` operations.

---

## 1. Problem Analysis: Bottlenecks

### 1.1 Arch Pattern

Most SOTA agent memory systems share a common architecture:

```
Raw Messages (base data)  →  Python App Server (view maintenance logic)  →  Database(s) (materialized views)
```

The app server transforms raw conversational msgs into structured **views** — knowledge graphs, entity summaries, community clusters — through a pipeline of LLM calls and (e.g.,) db operations. This is kinda like **incremental view maintenance**: each new message triggers updates to derived views.

Two asymmetric workloads:
- **Insertion flow**: Expensive, slow, accuracy-oriented. Dominates cost.
- **Retrieval flow**: Relatively fast and cheap. Hybrid search + reranking.

The insertion flow is the primary optimization target.

### 1.2 Bottleneck 1: Eager View Maintenance

**Concept**: Every incoming message triggers the *full* maintenance pipeline, regardless of information content.

In Graphiti, `add_episode(m, g, t)` executes 8 sequential phases for *every* message (see `data_flow_relational.md` §1):

| Phase | Operation | Type | Cost Driver |
|-------|-----------|------|-------------|
| 1 | Context Retrieval | DB | `τ_{created_at↓}(σ_{group_id=g}(Episodic))` [LIMIT 10] |
| 2 | Entity Extraction | **LLM** | `sem_map((m, E_prev), "Extract entity nodes...")` |
| 3 | Entity Deduplication | DB + **LLM** | Hybrid search [LIMIT 2k] per entity + `sem_join` |
| 4 | Edge Extraction | **LLM** | `sem_map((m, N, E_prev, t), "Extract relationships...")` |
| 5 | Edge Deduplication | DB + **LLM** | 2× hybrid search per edge + `sem_join` + `sem_filter` |
| 6 | Summary Generation | **LLM** | `sem_map` per entity (parallel) |
| 7 | Persist to DB | DB | MERGE nodes, edges, embeddings |
| 8 | Community Update | **LLM** | `sem_agg` + `sem_map` per entity (optional) |

**Source evidence** (`graphiti_core/graphiti.py`, lines 863–999):
```python
# Every add_episode triggers ALL of these — no short-circuit
extracted_nodes = await extract_nodes(...)          # Phase 2: LLM
nodes, uuid_map, _ = await resolve_extracted_nodes(...)  # Phase 3: DB + LLM
resolved_edges, invalidated_edges = await self._extract_and_resolve_edges(...)  # Phase 4-5: LLM
hydrated_nodes = await extract_attributes_from_nodes(...)  # Phase 6: LLM
# No information-gain filter. "Hi", "OK", "I see" all trigger full pipeline.
```

**Problem**: Do we need a *cost-based decision* on whether a message warrants full processing? In db terms, sota is eager maintenance without a cost model — every base-data change triggers full view refresh.

### 1.3 Bottleneck 2: Sequential LLM Operator Chain

**Concept**: The insertion pipeline is a *chain of dependent semantic operators* where each LLM call blocks the next.

E.g., in zep, the dependency graph is strictly sequential:
```
extract_nodes(m) → dedupe_nodes(N, C) → extract_edges(m, N) → dedupe_edges(F, C) → summarize(N)
      LLM              DB+LLM                LLM                 DB+LLM              LLM×|N|
```

Each arrow is a slow LLM round-trip. For a single msg with 2 entities and 1 edge:
- **Minimum 5 sequential LLM calls**: extract_nodes → dedupe_nodes → extract_edges → dedupe_edges → 2× summarize
- Plus embedding calls and multiple DB round-trips between phases

**Key observation**: `extract_nodes` (Phase 2) and `extract_edges` (Phase 4) both receive the same message context `(m, E_prev)` and perform semantically related extraction tasks — both are `sem_map` operators on overlapping input with different output schemas.

**Problem**: Borrow from Palimpzest: The pipeline is a *fixed physical plan* with no logical-to-physical optimization. No operator fusion or reordering is considered. In database terms, this is executing a query without a query optimizer.

### 1.4 Bottleneck 3: Asymmetric Deduplication Fast-Paths

Maybe this is just for Zep, cuz someone else may not have the dedup ops.

**Concept**: Deduplication is one of the most expensive per-item operation, and fast-path optimizations are only partially applied in Zep.

**Node deduplication** has a three-tier fast-path (`graphiti_core/utils/maintenance/dedup_helpers.py`, lines 198–247):
1. **Exact name match** → bypass LLM (deterministic)
2. **MinHash LSH + Jaccard ≥ 0.9** → bypass LLM (probabilistic)
3. **Low entropy names** → defer to LLM

This is effectively a *predicate pushdown*: cheap local computation filters out cases before the expensive `sem_join` operator.

**Edge deduplication** lacks this optimization (`graphiti_core/utils/maintenance/edge_operations.py`, lines 300–314):
- Only exact `(source_uuid, target_uuid, normalized_fact)` string match — no fuzzy similarity fast-path
- Every non-exact-match edge triggers: (a) get edges between same nodes, (b) hybrid search for related edges, (c) hybrid search for invalidation candidates, (d) LLM `sem_join` + `sem_filter`

**Per-edge cost in `data_flow_relational.md` §1 Phase 5**:
```
∀ f ∈ F_raw:
  E_between = σ_{src=f.src ∧ tgt=f.tgt}(RELATES_TO)          -- DB query 1
  C_f = RRF(FTS(...) [LIMIT k], cosine(...) [LIMIT k])       -- DB query 2 (related)
  I_f = RRF(FTS(...) [LIMIT k], cosine(...) [LIMIT k])       -- DB query 3 (invalidation)
  sem_join({f} × C_f, "identical?") ∧ sem_filter(I_f, "contradicts?")  -- LLM call
```

That is **3 DB queries + 1 LLM call per extracted edge**, with no probabilistic shortcut.

**Problem**: The fast-path pattern (cheap filter before expensive LLM) is proven effective for nodes but not generalized. A systematic approach to predicate pushdown across all dedup stages is missing.

Also, we need to test and think the about the reason, like accuracy loss sharply if we use this fast-path for edges?

### 1.5 Bottleneck 4: Brute-Force Vector Search in Deduplication

Maybe this is just a small engineering problem?

**Concept**: The deduplication phases (3, 5) and the retrieval flow both rely on vector cosine similarity computed via full scan — no approximate nearest neighbor (ANN) index.

**Source evidence** (`graphiti_core/search/search_utils.py`):
```python
# Lines 70-77: Brute-force cosine similarity
def calculate_cosine_similarity(vector1, vector2):
    dot_product = np.dot(vector1, vector2)
    ...

# Lines 371-379: Used in loops over ALL candidates
for r in resp:
    if r['embedding']:
        score = calculate_cosine_similarity(
            search_vector, list(map(float, r['embedding'].split(',')))
        )
```

Neo4j stores vectors via `db.create.setNodeVectorProperty()` but **no HNSW vector index is created** (`CREATE VECTOR INDEX` is never called). Every cosine similarity search is O(n) over all entities/edges in the group.

In `data_flow_relational.md`, all vector operations follow this pattern:
```
τ_{score↓}(σ_{score>0.6}(Entity ⊗_{cosine(name_embedding, embed(n.name))})) [LIMIT 2k]
```
The `⊗` (cross product with score) is computed by scanning all rows — there is no index-based pruning.

**Problem**: As graph size grows, every dedup hybrid search becomes O(|V|) or O(|E|). This creates a *quadratic interaction*: more messages → more entities → slower dedup per message.

### 1.6 Bottleneck 5: Redundant Hybrid Searches in Edge Deduplication

Maybe also a minor engineering issue.

**Concept**: Edge deduplication issues two separate hybrid searches per edge that share the same query and similar search space.

From `data_flow_relational.md` §1 Phase 5, for each extracted edge `f`:
1. **Related edge search** (filtered by `E_between`): `RRF(FTS('edge_name_and_fact', f.fact, g), cosine(fact_embedding, f.embedding))`
2. **Invalidation search** (unfiltered): `RRF(FTS('edge_name_and_fact', f.fact, g), cosine(fact_embedding, f.embedding))`

Both searches use **the same query** (`f.fact`), **the same FTS index**, and **the same embedding**. The only difference is the filter: search 1 restricts to edges between the same source/target nodes; search 2 has no such filter.

**Source evidence** (`graphiti_core/utils/maintenance/edge_operations.py`, lines 328–360)

**Problem**: The unfiltered search (2) is a strict superset of the filtered search (1). This is a missed *common subexpression elimination* — a single broader search could serve both purposes with post-hoc filtering.

### 1.7 Bottleneck 6: No Workload-Aware Execution Planning

**Concept**: The system executes the same physical plan regardless of workload characteristics (graph size, message density, R/W ratio, LLM latency).

Observations:
- **Graph size invariance**: With 10 entities or 10,000 entities, the same `[LIMIT 2k]` hybrid search is used for dedup candidate retrieval. No adaptive limit based on graph density.
- **Message density invariance**: A message mentioning 20 entities (dense, e.g., product catalog) and a message with 1 entity (sparse, e.g., greeting) both go through the same pipeline. (Note: Graphiti does have `add_episode_bulk` for explicit batching, and automatic chunking for high-density content, but no *adaptive* plan selection per message.)
- **No runtime cost model**: The system has no mechanism to estimate whether an LLM call is worth its cost. For example, summarizing an entity whose summary will not change is wasted work.

**Problem**: In database terms, there is no query optimizer — only a single hardcoded physical plan. A cost-based optimizer could select between plans (e.g., skip summary if entity summary hasn't changed, skip community update if entity is already in a stable community).

### 1.8 Summary: Mapping Bottlenecks to Semantic Operators

The insertion flow can be decomposed into LOTUS-like semantic operators:

| Bottleneck | Operator(s) Involved | Database Analogy |
|------------|---------------------|------------------|
| Eager maintenance | Entire pipeline | Eager vs. lazy/deferred view maintenance |
| Sequential operators | `sem_map` → `sem_join` → `sem_map` → `sem_join`+`sem_filter` → `sem_map` | Fixed query plan without optimization |
| Asymmetric fast-paths | `sem_join` (node dedup has pushdown; edge dedup does not) | Selective predicate pushdown |
| Brute-force vector | `⊗_{cosine}` in every hybrid search | Missing index (no HNSW) |
| Redundant searches | Two `search()` calls per edge with same query | Missing common subexpression elimination |
| No workload awareness | All operators use fixed parameters | No cost-based optimizer |

These are *system-level* problems — they affect latency, throughput, and cost regardless of the accuracy of any individual LLM call. They are also *general*: any agent memory system with LLM-based view maintenance will face analogous bottlenecks.

---

## 2. Available Resources in the Codebase

All paths are relative to the repository root (`zep-repos/`).

### 2.1 Datasets

| Dataset | Path | Format | Scale | Key Properties |
|---------|------|--------|-------|----------------|
| **LongMemEval** | `zep-graphiti/tests/evals/data/longmemeval_data/longmemeval_oracle.json` | JSON array | 500 questions, 940 unique sessions, 10,866 messages | Ground-truth Q&A pairs. 6 question types: temporal-reasoning (133), multi-session (133), knowledge-update (78), single-session-user (70), single-session-assistant (56), single-session-preference (30). Sessions per entry: 1–6 (avg 1.9). Messages per entry: 2–72 (avg 21.9). |
| **Wizard of Oz** | `zep-graphiti/examples/wizard_of_oz/woo.txt` | Plain text | 4,671 lines, 210 KB, 24 chapters | Long-form narrative. High entity density (characters, locations). Good for scalability curve testing. |
| **Podcast** | `zep-graphiti/examples/podcast/podcast_transcript.txt` | Plain text | 418 lines, 43 KB | Multi-speaker conversational transcript. Moderate entity density. |
| **E-commerce** | `zep-graphiti/examples/data/manybirds_products.json` | JSON | 42 KB | Structured product data. High entity density per record. |
| **Eval Harness Conversations** | `zep/zep-eval-harness/data/conversations/*.json` | JSON | 2 files, ~100 lines total (1 user, 2 conversations) | Small hand-crafted conversations with 4 test cases (`zep/zep-eval-harness/data/test_cases/`). |

### 2.2 Evaluation Frameworks

| Framework | Path | Architecture | What It Measures |
|-----------|------|-------------|------------------|
| **Graphiti E2E Eval** | `zep-graphiti/tests/evals/eval_e2e_graph_building.py` | Builds graph from LongMemEval sessions → LLM-as-Judge compares candidate vs baseline | Graph building quality. Uses `build_subgraph()` per user + `eval_graph()` with `gpt-4.1-mini` baseline. 181 lines. |
| **Zep LongMemEval** | `zep/benchmarks/longmemeval/zep_longmem_eval.py` | Ingestion → graph search → response generation → LLM grading | End-to-end memory quality. Has per-question-type grading prompts. 592 lines. Targets **Zep Cloud API** (not local Graphiti). |
| **Zep LOCOMO** | `zep/benchmarks/locomo/` | Unified CLI (`benchmark.py`) → ingestion (`ingestion.py`) → evaluation (`evaluation.py`) → persistence (`persistence.py`) | Comprehensive metrics: `LatencyStats` (p50/p90/p95/p99), `TokenStats`, `CategoryMetrics`, `CompletenessGrade`. 6 experiment runs saved. Targets **Zep Cloud API**. |
| **Zep Eval Harness** | `zep/zep-eval-harness/` | Ingest conversations (`zep_ingest.py`) → evaluate with graph search + LLM judge (`zep_evaluate.py`) | Custom ontology support. Configurable search limits (FACTS=20, ENTITIES=10). Targets **Zep Cloud API**. |

### 2.3 Stress Testing

| Tool | Path | Capabilities |
|------|------|--------------|
| **MCP Stress Tests** | `zep-graphiti/mcp_server/tests/test_stress_load.py` | 7 load scenarios: sustained load, spike load, memory leak detection, connection pool exhaustion, gradual degradation, large payload handling, rate limit handling. Configurable: `num_clients`, `operations_per_client`, `ramp_up_time`, `test_duration`, `target_throughput`. Collects `LoadTestResult` with p50/p95/p99 latency. 528 lines. Targets **MCP server** (not direct Graphiti API). |

### 2.4 Trace / Profiling Data

From the OTEL notebook (`agent_conversation_full_trace.ipynb`), the following trace logs exist:

| File | Lines | Content |
|------|-------|---------|
| `trace_llm.jsonl` | 54 | All LLM calls: model, messages (full prompt), params, usage (tokens), finish_reason |
| `trace_neo4j.jsonl` | 4 | Neo4j read queries with Cypher and results |
| `trace_neo4j_write.jsonl` | — | Neo4j write operations |
| `trace_embeddings.jsonl` | 56 | Embedding calls with input text and output vectors |
| `trace_main.jsonl` | 27 | High-level span data (function names, durations) |
| `trace_parallel.jsonl` | — | Parallel execution spans |

All located in `zep-graphiti/examples/neo4j_otel/`. Collected from a 3-turn conversation demo.

### 2.5 Example Runners (Ingestion Scripts)

| Script | Path | What It Does |
|--------|------|--------------|
| **Wizard of Oz runner** | `zep-graphiti/examples/wizard_of_oz/runner.py` | Parses chapters from `woo.txt` via `parser.py`, ingests each chapter as an episode. Direct Graphiti API. |
| **Podcast runner** | `zep-graphiti/examples/podcast/podcast_runner.py` | Parses transcript via `transcript_parser.py`, ingests each segment. Direct Graphiti API. |
| **Dense vs Normal** | `zep-graphiti/examples/quickstart/dense_vs_normal_ingestion.py` | Demonstrates chunking behavior for high-density content (product catalogs) vs low-density (prose). Shows `CHUNK_MIN_TOKENS`, `CHUNK_DENSITY_THRESHOLD`, `CHUNK_TOKEN_SIZE` env vars. 343 lines. |
| **Quickstart** | `zep-graphiti/examples/quickstart/quickstart_neo4j.py` | Minimal example of Graphiti init + add_episode + search. |

### 2.6 Important Caveats

- **Zep Cloud vs local Graphiti**: The benchmarks in `zep/benchmarks/` and `zep/zep-eval-harness/` target the **Zep Cloud API** (`zep_cloud.client.AsyncZep`), not the local `graphiti_core` library. They cannot be run directly against a local Graphiti+Neo4j setup without adaptation.
- **LongMemEval in Graphiti**: The `eval_e2e_graph_building.py` in `zep-graphiti/tests/evals/` *does* use local Graphiti directly, but it's designed for accuracy evaluation (LLM-as-Judge), not performance profiling.
- **MCP Stress Tests**: Target the MCP server layer, which adds HTTP/protocol overhead. For pure Graphiti profiling, direct API calls are more appropriate.
- **Trace data**: The existing JSONL traces cover only a 3-turn conversation. Larger-scale traces need to be collected.

---

## 3. Test Case Strategy

### 3.1 Priority 0: Per-Message Cost Breakdown (OTEL Notebook Extension)

**Goal**: Quantify *what percentage of insertion cost is wasted* on low-information messages.

**Method**:
1. Take a single LongMemEval entry (e.g., entry 0: 3 sessions, 36 messages, temporal-reasoning type).
2. Ingest all 36 messages via `add_episode()` with full OTEL tracing (extend `agent_conversation_full_trace.ipynb`).
3. For each message, record:
   - Total wall-clock time
   - Number of LLM calls and token consumption
   - Number of entities/edges extracted (|N_raw|, |F_raw|)
   - Number of entities/edges that were duplicates
   - Number of new nodes/edges actually persisted
4. Classify each message as **informative** (|N_raw| > 0 or |F_raw| > 0) vs **low-information** (zero extraction yield).
5. Compute: `waste_ratio = cost(low-information messages) / cost(all messages)`.

**Expected finding**: A significant fraction of messages (assistant acknowledgments, greetings, fillers) still trigger full pipeline but produce zero new graph state.

**Dataset**: `zep-graphiti/tests/evals/data/longmemeval_data/longmemeval_oracle.json` (entry 0).

### 3.2 Priority 1: Scalability Curve (Graph Size vs. Insertion Latency)

**Goal**: Measure how insertion latency grows as the knowledge graph scales.

**Method**:
1. Use Wizard of Oz (24 chapters, 210KB). Ingest chapters sequentially.
2. After each chapter, record:
   - Graph size: |Entity|, |RELATES_TO|, |Episodic|
   - `add_episode` latency for the chapter
   - Breakdown: time in LLM calls vs. DB queries (especially hybrid search)
3. Plot: graph size (x-axis) vs. insertion latency (y-axis).
4. Separate LLM time from DB time to identify which component scales worse.

**Expected finding**: DB query time (especially brute-force cosine similarity in dedup) grows with graph size, while LLM time stays roughly constant per message. The crossover point indicates when vector indexing becomes critical.

**Dataset**: `zep-graphiti/examples/wizard_of_oz/woo.txt` + `runner.py`.

### 3.3 Priority 2: LLM Prompt Overlap Analysis (Offline)

**Goal**: Quantify the prompt overlap between `extract_nodes` and `extract_edges` to estimate operator fusion potential.

**Method**:
1. Parse `trace_llm.jsonl` (54 entries from the 3-turn demo).
2. For each turn, extract the system+user prompts for:
   - `extract_nodes` call (Phase 2)
   - `extract_edges` call (Phase 4)
3. Compute token overlap: `overlap = |tokens(prompt_nodes) ∩ tokens(prompt_edges)| / |tokens(prompt_nodes) ∪ tokens(prompt_edges)|`
4. Estimate fused-call token savings: `savings = |tokens(prompt_nodes)| + |tokens(prompt_edges)| - |tokens(fused_prompt)|`

**Expected finding**: High overlap (>70%) because both calls include the same message `m` and previous episodes `E_prev`. Fusion could save ~40-50% of input tokens for extraction phases.

**Dataset**: `zep-graphiti/examples/neo4j_otel/trace_llm.jsonl` (existing, no new runs needed).

### 3.4 Test Case A: Low-Information Message Stream

**Goal**: Directly measure the cost of eager maintenance on a worst-case message stream.

**Method**:
1. Construct a synthetic conversation: 50% informative messages, 50% low-information ("ok", "I see", "hmm", "thanks", "got it").
2. Ingest the full stream. Measure total cost.
3. Ingest only the informative messages. Measure total cost.
4. Compare: graph state should be identical; cost difference = waste.

**Bottleneck targeted**: §1.2 (Eager maintenance). Directly quantifies the upper bound of savings from information-gain filtering.

**Dataset**: Synthetic (constructed from LongMemEval messages by interleaving fillers).

### 3.5 Test Case B: Read/Write Ratio Impact

**Goal**: Measure how a mixed workload (concurrent reads and writes) affects insertion latency.

**Method**:
1. Ingest Wizard of Oz chapters while issuing concurrent `search()` queries at different rates.
2. Vary R/W ratio: 0:1 (write-only), 1:1, 5:1, 10:1.
3. Measure: insertion latency, search latency, and Neo4j lock contention (if any).

**Bottleneck targeted**: §1.7 (No workload awareness). In lazy maintenance, high read pressure would trigger view materialization; in eager maintenance, reads and writes are independent — this test verifies whether that independence holds under load.

**Dataset**: `zep-graphiti/examples/wizard_of_oz/woo.txt` + synthetic search queries.

### 3.6 Test Case C: Sequential LLM Cost Breakdown

**Goal**: Measure the latency contribution of each pipeline phase to identify the dominant bottleneck.

**Method**:
1. Extend P0 tracing to capture per-phase wall-clock time for each message.
2. Aggregate across all messages in a LongMemEval entry.
3. Produce a stacked bar chart: Phase 2 (extract_nodes) + Phase 3 (dedupe_nodes) + Phase 4 (extract_edges) + Phase 5 (dedupe_edges) + Phase 6 (summarize) + Phase 7 (persist) + Phase 8 (community).
4. Also break down: LLM wait time vs. DB query time vs. embedding time vs. Python compute time.

**Bottleneck targeted**: §1.3 (Sequential operators). Identifies which phase dominates and thus which optimization has the highest ROI.

**Dataset**: Same as P0 (LongMemEval entry 0, 36 messages).

### 3.7 Test Case D: Deduplication Scalability

**Goal**: Isolate the dedup cost as a function of graph size.

**Method**:
1. Pre-populate graphs of varying sizes: 50, 200, 500, 1000, 2000 entities.
2. Insert a fixed set of 10 messages into each graph.
3. Measure dedup latency (Phase 3 + Phase 5) per message.
4. Plot: graph size (x-axis) vs. dedup latency (y-axis).
5. Separate: node dedup (has MinHash fast-path) vs. edge dedup (no fast-path).

**Bottleneck targeted**: §1.4 (Asymmetric fast-paths) + §1.5 (Brute-force vector search). Expected result: edge dedup latency grows faster than node dedup latency because it lacks the MinHash bypass.

**Dataset**: Wizard of Oz (progressive ingestion to build graphs of target sizes).

### 3.8 Test Case E: Entity Drift (Knowledge Update)

**Goal**: Measure the cost of processing messages that update existing facts (entity drift / knowledge update).

**Method**:
1. From LongMemEval, select entries with `question_type = 'knowledge-update'` (78 entries). These represent conversations where facts change over time (e.g., "I moved from NYC to SF").
2. Ingest the haystack sessions for a few selected entries.
3. Track: how many edges get invalidated (`F_inv` in Phase 5), how many summaries change (Phase 6).
4. Compare insertion cost: knowledge-update entries vs. single-session entries.

**Bottleneck targeted**: §1.6 (Redundant searches). Knowledge-update messages are the most expensive because they trigger both the dedup path AND the invalidation path — both hybrid searches in Phase 5 will find matches.

**Dataset**: LongMemEval `knowledge-update` entries (78 available).

### 3.9 Test Case F: Mixed Workload Profile (End-to-End Benchmark)

**Goal**: Establish a comprehensive baseline for the full system under realistic conditions.

**Method**:
1. Use multiple LongMemEval entries covering all 6 question types.
2. Ingest all haystack sessions (using the `eval_e2e_graph_building.py` `build_subgraph` pattern).
3. After ingestion, run the corresponding questions as `search()` queries.
4. Record end-to-end metrics: total ingestion time, total search time, total LLM tokens, total DB queries.
5. Grade retrieval quality using the existing LLM-as-Judge framework.

**Purpose**: This is the *baseline measurement* that any optimization must preserve. It combines ingestion profiling with quality verification — ensuring that optimizations don't degrade retrieval accuracy.

**Dataset**: LongMemEval (select 10–50 entries across all question types for manageable runtime).

### 3.10 Execution Order Summary

| Priority | Test | Primary Question |
|----------|------|-----------------|
| **P0** | §3.1 Per-message cost breakdown | How much insertion work is wasted? |
| **P1** | §3.2 Scalability curve | Does insertion latency grow with graph size? Where? |
| **P2** | §3.3 Prompt overlap analysis | How much can operator fusion save? |
| **A** | §3.4 Low-info message stream | Upper bound of lazy maintenance savings |
| **B** | §3.5 R/W ratio impact | Does concurrent search affect insertion? |
| **C** | §3.6 Phase cost breakdown | Which phase dominates? |
| **D** | §3.7 Dedup scalability | Node dedup vs edge dedup scaling behavior |
| **E** | §3.8 Knowledge update cost | Cost of fact invalidation |
| **F** | §3.9 End-to-end baseline | Baseline quality + performance numbers |